## 5.6 章节实践

通过本章的系统学习，我们掌握了跳频分集的原理、六阶段接入建链流程和闭环功率自适应的核心策略。现提供以下综合实践练习：

**跳频 + AMC 联合仿真**，在 Rayleigh 衰落下同时实现跳频分集和 MCS 自适应跟踪，补全 4 处空缺。

要求：

1. 补全跳频信道系数 h 的获取
2. 补全 LinkQualityTracker 的 CRC 记录
3. 补全 MCS 调整决策
4. 补全下一帧 MCS 的读取

完成后运行 `python chapter5_practice.py`，观察 MCS 随窗口 FER 的动态调整轨迹。

In [ ]:
%%writefile chapter5_practice.py
import sys
sys.path.insert(0, "../src")
import numpy as np
import matplotlib.pyplot as plt
from nearlink_sdr.common.polar import PolarEncoder, get_polar_decoder
from nearlink_sdr.phy.channel import ChannelConfig, ChannelModel
from nearlink_sdr.mac.qos import LinkQualityTracker
from nearlink_sdr.phy.tx_pipeline import TxConfig
from nearlink_sdr.phy.mac_interface import iq_to_mac, mac_to_iq

snr_db = 4.0
n_frames = 100
code_n, K = 256, 112
snr_lin = 10.0 ** (snr_db / 10.0)

rng = np.random.default_rng(42)
enc = PolarEncoder(code_n, K)
dec = get_polar_decoder(code_n, K)

tracker = LinkQualityTracker(window_size=16)
tracker._current_mcs = 5
mcs_hist = []

for i in range(n_frames):
    # ---- TxConfig 使用当前 MCS ----
    mcs_idx = tracker.current_mcs
    cfg = TxConfig(frame_type=2, mcs_index=mcs_idx, pid=0x123456,
                   whitening_seed=0x52, crc_seed=0x555555,
                   crc_len=24, ctrl_bits_len=28, pilot_interval=8)

    # ---- 发送: 随机载荷 → mac_to_iq ----
    mac_payload = bytes(rng.integers(0, 256, 10, dtype=np.uint8))
    from nearlink_sdr.mac.frame import AsyncDataFrame
    frame = AsyncDataFrame(segment_type=0, data=mac_payload)
    mac_bytes = frame.pack()
    iq = mac_to_iq(mac_bytes, cfg)

    # ---- 跳频: 每帧独立 Rayleigh 信道 ----
    ch = ChannelModel(config=ChannelConfig(
        snr_db=snr_db, channel_type="rayleigh",
        seed=int(rng.integers(0, 2**31))))
    # ==== 1: 获取跳频信道系数（1行）====
    h_hop = ______________

    # ---- AWGN + 检测 ----
    rx_iq = iq * h_hop
    noise_var = 1.0 / (2.0 * snr_lin)
    noise = np.sqrt(noise_var) * (rng.standard_normal(len(iq))
                                  + 1j * rng.standard_normal(len(iq)))
    rx_iq = rx_iq + noise
    rx = iq_to_mac(rx_iq, cfg, len(mac_bytes))

    # ==== : 补全 AMC 闭环（3处空缺）====
    #  2: 记录 CRC 结果
    ______________
    #  3: 应用 MCS 调整
    ______________

    mcs_hist.append(tracker.current_mcs)

# ---- MCS 轨迹图 ----
plt.step(range(n_frames), mcs_hist, where="mid", "b-", lw=1.5)
plt.xlabel("Frame Index"); plt.ylabel("MCS Index")
plt.yticks(range(13))
plt.title(f"Hopping + AMC (SNR={snr_db:.0f} dB, ws={tracker.window_size})")
plt.grid(True, ls="--", alpha=0.5); plt.show()

print(f"MCS range: {min(mcs_hist)} -> {max(mcs_hist)}")
print(f"Final MCS: {mcs_hist[-1]}")
print("MCS 自适应调整次数:", sum(1 for i in range(1, len(mcs_hist)) if mcs_hist[i] != mcs_hist[i-1]))

执行以下命令进行编译并验证结果：

In [ ]:
%run chapter5_practice.py

执行以下代码获取答案

In [ ]:
!cat answer/05.06_answer.txt